## ▶ What you'll see when you run this
- The **same prompt** answered side by side by Claude, Gemini, and a local model, each with its **response time** printed.

**Time:** ~15 min · **Cost:** free (defaults to cheapest model: Gemini Flash / Claude Haiku / local Ollama) · **Keys:** **ANTHROPIC_API_KEY** and/or **GEMINI_API_KEY** (local Ollama needs none)

# Week 8 · Notebook 2 — Claude vs Gemini vs Local
**CSCI 250 · Fall 2026**

Send the **same prompt** to three model sources and compare length, tone, correctness, and speed:
- **Anthropic Claude** (cloud)
- **Google Gemini** (cloud)
- **Ollama** (local, no key)

> **Keys:** Colab 🔑 *Secrets* + `userdata.get('NAME')`; locally use env vars. **Never commit keys.**

## 0. A no-key warm-up (runs instantly)
Before any API calls, confirm the notebook is alive.

In [ ]:
print('Notebook ready. We will send ONE shared prompt to 3 models below.')
print('Cheapest defaults: Gemini Flash / Claude Haiku / local Ollama.')

## 0b. Install SDKs

In [ ]:
!pip -q install anthropic google-generativeai ollama

## 1. Load API keys safely

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except Exception:
    pass  # locally, set these in your shell environment
print('Anthropic key set:', bool(os.environ.get('ANTHROPIC_API_KEY')))
print('Gemini key set:', bool(os.environ.get('GEMINI_API_KEY')))

## 2. The shared prompt + a timing helper

In [ ]:
import time
PROMPT = ('In exactly 3 bullet points, explain what a context window is to a '
          'new CS student.')

def timed(fn):
    t0 = time.time()
    out = fn()
    return out, round(time.time() - t0, 2)

## 3. Anthropic Claude

In [ ]:
import anthropic
client = anthropic.Anthropic()

def call_claude():
    m = client.messages.create(model='claude-sonnet-4-6', max_tokens=200,
        temperature=0.3, messages=[{'role':'user','content':PROMPT}])
    return m.content[0].text

try:
    out, secs = timed(call_claude)
    print(f'CLAUDE ({secs}s):\n{out}')
except Exception as e:
    print('Claude error:', e)

## 4. Google Gemini

In [ ]:
import google.generativeai as genai
genai.configure(api_key=os.environ.get('GEMINI_API_KEY', ''))
gem = genai.GenerativeModel('gemini-2.5-flash')

def call_gemini():
    r = gem.generate_content(
        PROMPT,
        generation_config={'temperature': 0.3, 'max_output_tokens': 200})
    return r.text

try:
    out, secs = timed(call_gemini)
    print(f'GEMINI ({secs}s):\n{out}')
except Exception as e:
    print('Gemini error:', e)

## 5. Local model with Ollama
Install Ollama from **ollama.com**, then in a terminal: `ollama pull llama3.2`.
In Colab you can instead run:
```
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve >/tmp/ollama.log 2>&1 &
!ollama pull llama3.2
```

In [ ]:
import ollama

def call_ollama():
    r = ollama.chat(model='llama3.2',
                    messages=[{'role':'user','content':PROMPT}])
    return r['message']['content']

try:
    out, secs = timed(call_ollama)
    print(f'OLLAMA llama3.2 ({secs}s):\n{out}')
except Exception as e:
    print('Ollama not running yet:', e)

## 6. Compare (fill this in for S1)
| Model | Length | Tone | Correct? | Speed (s) |
|---|---|---|---|---|
| Claude (claude-sonnet-4-6) |  |  |  |  |
| Gemini (gemini-2.5-flash) |  |  |  |  |
| Local (llama3.2 via Ollama) |  |  |  |  |

Then write ~150 words: which model would you pick for a **privacy-sensitive** task, for a **quick free** task, and for the **highest-quality** answer — and why?

In [ ]:
# notes:
